# Phase 3.3 · Causal query-pathway interventions

Phases 3.1–3.2 were **observational**: they localized within-KV-group sink
divergence to the query pathway. Phase 3.3 is **causal** — it re-runs inference
under counterfactual queries and asks whether sink behaviour *moves with the
query*.

Within one KV group every head reads the **same K and V**, so any change confined
to the query is the only thing that can move sink behaviour. That is what licenses
the causal reading, and why every intervention here stays inside a single KV group.

Three experiments:

* **A — W_Q weight swap.** Exchange the query-projection row-blocks of two
  same-group heads and remeasure.
* **B — Query activation patching.** Overwrite one head's query activation with
  another's via a forward hook (transfer measurement), or swap them (equivalence
  check against A).
* **C — Query ablation.** Zero / scale / normalize / add noise to a head's query
  to dissect which part of the query carries the sink signal.

Unlike 3.2, **this notebook loads Qwen3-1.7B** and runs forward passes.

## The intervention, precisely

The Qwen3 attention path (verified against the installed `transformers`) is:

```
query_states = q_norm( q_proj(hidden).view(B, T, Hq, D) ).transpose(1,2)   # [B, Hq, T, D]
query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)
```

Three facts make the swap clean:

1. `q_proj` has **no bias** (`attention_bias=false`), so `q = W_Q · x` — purely linear.
2. `q_norm` is a **single RMSNorm over `head_dim` with one shared γ**, applied
   identically to every head.
3. RoPE depends only on **position**, not head index.

So every operation after the per-head query slice is **head-independent**.
Exchanging the query *source* of a head therefore propagates untouched through
norm and RoPE. This is why **editing `W_Q` rows (A) and copying the `q_proj`
activation slice (B) produce the **identical** `query_states`** — they are the same
intervention at two stages, and their agreement (verified to `0.0e+00`) is an
internal causal check.

One subtlety used in Experiment C: because the patch is applied *before* `q_norm`
and RMSNorm is **scale-invariant**, a pure pre-norm rescale is nearly inert — the
norm re-normalizes it away. Zeroing (0 → 0), re-normalizing (direction only), and
noising (direction corrupted) *do* move behaviour; scaling barely does. That
contrast is itself informative about what the query carries.

## Experimental Design and Statistical Pooling

Every pooling decision is fixed **before** any number is produced, because for a
causal claim the order of averaging can manufacture or hide an effect.

**What is a single experimental sample?** One query token: a specific
`(layer L, head h, query position t)` in one prompt `p`, under one condition. The
causal unit is the *paired* sample — the same `(L, h, t, p)` under baseline and
under intervention — since the intervention changes only the query there, holding
prompt, position, keys and values fixed.

**Per token, per head, or per sequence?** Per **token**, always, and first. Every
metric is a token-level primitive; head- and sequence-level numbers are strictly
aggregates of those primitives, never computed directly.

**How are sequence lengths handled?** Prompts differ in length, so we never pool
raw tokens across prompts (a long prompt would dominate). We reduce to a
per-sequence mean first (Level 2); each prompt then carries equal weight.

**How are BOS tokens excluded / included?** BOS (position 0, the sink itself) is
**included as a key** — it is what we measure attention toward — but **excluded as
a query** (a query at position 0 has no competitors). We also drop query positions
below `min_query_pos` (default 4) to avoid the `log t` transient. Both rules apply
at Level 1.

**Is sink score averaged over query positions?** Yes: sink *probability* is the
per-token BOS probability; sink *score* is its mean over a head's valid tokens
within a sequence (Level 2), then over sequences (Level 3).

**Are logits averaged before or after taking margins?** **After.** BOS advantage
and LSE margins are computed at the token level, then averaged. Averaging logits
first and subtracting later would inject Jensen artifacts.

**How is head swapping paired across prompts?** The same ordered `(recipient,
donor)` is applied to every prompt, each run under both baseline and intervention.
Comparisons are paired at `(L, h, t, p)` — identical prompt, position, K and V,
only the query differs.

**How many prompts contribute to each statistic?** All `N` prompts of the frozen
benchmark subset used in 3.1/3.2, so results are comparable across phases. Each
per-head (Level 3) statistic is a mean over `N` sequences, reported with its
across-sequence spread.

**Which dimensions are averaged first, and why?** Strict order **token → sequence
→ head → overall**: tokens within a sequence first (length-independent), then
sequences within a head (sampling-independent), then heads (global summary only).
Reporting lives at **Level 3 (per head)**, the granularity of the causal claim.

**Headline statistic — transfer fraction.** For each metric, at Level 3,

$$\text{transfer}=\frac{m^{\text{int}}_{\text{recip}}-m^{\text{base}}_{\text{recip}}}
{m^{\text{base}}_{\text{donor}}-m^{\text{base}}_{\text{recip}}}$$

0 = recipient unmoved, 1 = recipient lands on the donor; undefined (flagged) when
the two baselines coincide. A single-layer *full* query transplant makes the
recipient's attention at that layer identical to the donor's, so transfer ≈ 1 **by
construction** — the scientifically interesting signal on the real model lies in
the downstream ripple and in the partial effects of the ablations, not in that
tautological 1.

## Part 0 · Setup

In [ ]:
import sys, os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/attention_sink_project')
else:
    BASE = Path('.')

# discover the helper modules (both are needed: phase33 imports phase3)
for cand in [BASE, Path('/content'), Path('.')]:
    if (cand / 'phase33_utils.py').exists():
        sys.path.insert(0, str(cand)); break
for cand in [BASE, Path('/content'), Path('.')]:
    if (cand / 'phase3_utils.py').exists():
        sys.path.insert(0, str(cand)); break

import phase3_utils as U
import phase33_utils as P3
import numpy as np, torch, pandas as pd
print('phase3_utils + phase33_utils loaded')

# --- Configuration -----------------------------------------------------------
# The prompt-construction knobs are IDENTICAL to Phase 3.1 (and thus 3.2), so the
# only thing differing between phases is what is measured, not what it is measured
# on. Do not introduce a fallback corpus here: results must trace to the same
# frozen benchmark rows as Phases 1/2/3.1/3.2.
CONFIG = dict(
    MODEL_NAME    = 'Qwen/Qwen3-1.7B',
    PROMPT_MODE   = 'category_block',   # same construction as 3.1 / 3.2
    LANGUAGES     = ('eng', 'vie'),     # the benchmark is parallel -- keep both
    CATEGORIES    = None,               # None = all 5 categories
    MAX_SEQUENCES = None,               # None = all sequences; cap for a quick run
    STRICT_BENCHMARK = True,            # fail on any benchmark validation warning
    MAX_LEN       = 256,                # capture memory ~ T^2; matches 3.1
    DTYPE         = 'float32',          # margins compared at the 0.1-nat level
    PREPEND_BOS   = True,               # Qwen3's tokenizer does NOT add BOS itself
    MIN_QUERY_POS = 4,                  # drop early positions (competitor set too small)
    SEED          = 0,
)
torch.manual_seed(CONFIG['SEED'])

OUT_DIR = BASE / 'results' / 'phase3' / 'phase3_3_causal'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('writing Phase 3.3 outputs to:', OUT_DIR)

### Load Qwen3-1.7B

This notebook needs the live model. On the real checkpoint the interventions
produce meaningful numbers; the architecture facts above hold regardless of
weights.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

DTYPE = {'float32': torch.float32, 'bfloat16': torch.bfloat16,
         'float16': torch.float16}[CONFIG['DTYPE']]
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

t0 = time.time()
tok = AutoTokenizer.from_pretrained(CONFIG['MODEL_NAME'])
try:                                    # transformers renamed torch_dtype -> dtype
    model = AutoModelForCausalLM.from_pretrained(
        CONFIG['MODEL_NAME'], dtype=DTYPE, attn_implementation='eager')
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(
        CONFIG['MODEL_NAME'], torch_dtype=DTYPE, attn_implementation='eager')
model = model.to(DEVICE).eval()

c = model.config
top = P3.model_topology(model)
print(f"loaded {CONFIG['MODEL_NAME']} in {time.time()-t0:.1f}s on {DEVICE} / {CONFIG['DTYPE']}")
print(f"  layers={c.num_hidden_layers}  query heads={top['n_heads']}  "
      f"KV heads={top['n_kv']}  n_rep={top['n_rep']}  head_dim={top['head_dim']}")
print(f"  attn impl : {c._attn_implementation}   QK-norm: "
      f"q_norm={hasattr(model.model.layers[0].self_attn,'q_norm')}")
print(f"  bos_token_id={c.bos_token_id}  "
      f"(tokenizer adds BOS: {tok('hi')['input_ids'][:1] == [c.bos_token_id]})")
print("  within-group head pairs:", P3.within_group_pairs(model))

### Build the prompt set — the SAME frozen benchmark as Phases 1/2/3.1/3.2

Phase 3 reuses the **same 50 sentence pairs** as Phase 2 — 10 each across the five
categories — so the only thing differing between phases is *what is measured*, not
*what it is measured on*. `load_benchmark` **raises** if the benchmark is absent;
there is deliberately **no fallback corpus** (substituting prompts would produce
figures that look fine and are comparable to nothing).

Tokenization is byte-for-byte the Phase 3.1 construction: `add_special_tokens=False`,
truncate to `MAX_LEN − 1`, then **explicitly prepend BOS** (Qwen3's tokenizer does
not add one, and BOS is the sink we measure). Each sequence records the benchmark
`pair_ids` it was built from, and the benchmark fingerprint is saved for provenance.

In [ ]:
# --- Frozen benchmark: same pairs as Phases 1/2/3.1/3.2 (raises; no fallback) --
SEARCH_ROOTS = [BASE, Path('/content'), Path('.')]
bm = U.load_benchmark(SEARCH_ROOTS, strict=CONFIG['STRICT_BENCHMARK'])
print('benchmark   :', bm['path'])
print('pairs       :', len(bm['pairs']))
print('categories  :', sorted({p['category'] for p in bm['pairs']}))
print('fingerprint :', bm['fingerprint'])
if bm['warnings']:
    print('warnings    :', bm['warnings'])

prompts_spec = U.build_prompts(
    bm['pairs'], mode=CONFIG['PROMPT_MODE'], languages=CONFIG['LANGUAGES'],
    categories=CONFIG['CATEGORIES'], max_sequences=CONFIG['MAX_SEQUENCES'])
print()
print(U.describe_prompts(prompts_spec, tokenizer=tok, max_len=CONFIG['MAX_LEN']))

# tokenize EXACTLY as Phase 3.1: no auto specials, truncate, then prepend BOS
def to_ids(text):
    ids = tok(text, return_tensors='pt', add_special_tokens=False)['input_ids']
    ids = ids[:, : CONFIG['MAX_LEN'] - (1 if CONFIG['PREPEND_BOS'] else 0)]
    if CONFIG['PREPEND_BOS'] and c.bos_token_id is not None:
        ids = torch.cat([torch.tensor([[int(c.bos_token_id)]], dtype=ids.dtype), ids], 1)
    return ids.to(DEVICE)

prompts = [to_ids(P.text) for P in prompts_spec]
seq_meta = [dict(seq=i, seq_id=P.seq_id, category=P.category, language=P.language,
                 pair_ids=P.pair_ids, seq_len=int(prompts[i].shape[1]))
            for i, P in enumerate(prompts_spec)]

import json as _json
(OUT_DIR / 'sequences.json').write_text(_json.dumps(
    {'benchmark_path': str(bm['path']), 'benchmark_fingerprint': bm['fingerprint'],
     'prompt_mode': CONFIG['PROMPT_MODE'], 'languages': list(CONFIG['LANGUAGES']),
     'sequences': seq_meta}, indent=2))
print(f"\n{len(prompts)} sequences tokenized (seq index -> benchmark rows in sequences.json)")
print("lengths:", [m['seq_len'] for m in seq_meta])
print("fingerprint matches 3.1/3.2 if identical to the value in their manifests:",
      bm['fingerprint'])

### Choose intervention sites

A **site** is `(layer, head_a, head_b)` with `a`,`b` in the same KV group. The
default sweep is a manageable subset — a depth profile for one representative pair
plus a few pairs at early/mid/late layers. Flip `FULL_SWEEP` to run all layers ×
all within-group pairs (28 × 8 = 224 sites; several minutes on GPU).

In [ ]:
L = model.config.num_hidden_layers
pairs = P3.within_group_pairs(model)            # 8 within-group pairs
FULL_SWEEP = False

if FULL_SWEEP:
    sites = [(layer, a, b) for layer in range(L) for (a, b) in pairs]
else:
    depth_pair = pairs[0]                        # depth profile for one pair, all layers
    sites = [(layer, depth_pair[0], depth_pair[1]) for layer in range(L)]
    for layer in (1, L // 2, L - 1):            # a few pairs at early / mid / late
        for (a, b) in pairs:
            if (layer, a, b) not in sites:
                sites.append((layer, a, b))

# ablation targets: one head per layer sampled across depth
ablation_targets = [(layer, pairs[0][0]) for layer in (1, L // 2, L - 1)]
print(f"{len(sites)} intervention sites; {len(ablation_targets)} ablation targets "
      f"(FULL_SWEEP={FULL_SWEEP})")

## Run experiments A + B + C

`run_phase33` executes the baseline once (shared by all sites), then Experiment A
(W_Q swap), Experiment B (copy-mode patch for transfer, plus a swap-mode patch for
the A≡B equivalence check), and Experiment C (ablations). It pools to four levels,
computes transfer fractions, writes tidy CSVs + figures, and returns everything.

In [ ]:
res = P3.run_phase33(
    model, prompts, sites=sites,
    ablation_targets=ablation_targets,
    out_dir=OUT_DIR, min_query_pos=CONFIG['MIN_QUERY_POS'],
    run_patch=True, run_ablation=True, make_figures=True, show=True)

print("baseline per-token rows :", len(res.baseline))
print("W_Q swap per-token rows :", len(res.wq_swap))
print("copy-patch per-token rows:", len(res.patch))
print("ablation per-token rows :", len(res.ablation))

### Statistical pooling — the four levels

Averaging proceeds **token → sequence → head → overall**. Below are the baseline
pooled tables; the same structure is produced for every condition.

In [ ]:
for name, dfp in res.pooled_baseline.items():
    print(f"\n=== {name}  ({dfp.shape[0]} rows) ===")
    display(dfp.head(6))

## Experiments A & B · does sink behaviour follow the query?

**Transfer fraction** at Level 3 (per head): how far the recipient moves from its
own baseline toward the donor's when it receives the donor's query. Measured from
the copy-mode patch (donor left intact), which is the informative direction; the
symmetric swap is reserved for the equivalence check.

In [ ]:
tf = res.transfer
sink_tf = tf[(tf['metric'] == 'sink_score') & tf['has_gap']]
print("sink-score transfer:  median={:.3f}   IQR=[{:.3f}, {:.3f}]   n={}".format(
    sink_tf['transfer'].median(),
    sink_tf['transfer'].quantile(.25), sink_tf['transfer'].quantile(.75), len(sink_tf)))

display(tf[tf['metric'] == 'sink_score'][
    ['site_layer','recipient','donor','baseline_recipient',
     'baseline_donor','intervened_recipient','transfer','has_gap']].head(12))

The **before/after** scatter and the **transfer-by-metric / transfer-through-depth**
figures are written to `figures/` and shown inline above from `run_phase33`.

### Consistency check · A ≡ B

The weight swap and the swap-mode activation patch are the same intervention at
different stages; they must agree to floating-point.

In [ ]:
print(res.observations.get('consistency', '(patch not run)'))

## Experiment C · query ablations

Dissecting the query: zero it, rescale it, unit-normalize it (direction only), or
add noise. Recall that pre-norm **scaling is nearly inert** (QK-RMSNorm is
scale-invariant), so a flat scale response is expected and confirms magnitude
alone carries little of the sink signal.

In [ ]:
ab_l3 = P3.pool_levels(res.ablation)['L3_per_head'] if len(res.ablation) else None
if ab_l3 is not None:
    base_mean = res.pooled_baseline['L3_per_head'].rename(
        columns={'bos_prob': 'sink_score'})['sink_score'].mean()
    summary = (ab_l3.rename(columns={'bos_prob': 'sink_score'})
               .groupby('condition')['sink_score'].mean()
               .rename('mean_sink_score').reset_index())
    summary['vs_baseline'] = summary['mean_sink_score'] - base_mean
    print(f"baseline mean sink score: {base_mean:.4f}")
    display(summary)

## Interpretation

Data-driven conclusions from the measured transfer and ablations (no hard-coded
numbers). On the real checkpoint the single-layer transfer is expected near 1; the
scientific weight is in how transfer varies with **depth** and in the **ablation
contrasts**.

In [ ]:
import json
for key in ('transfer', 'consistency', 'ablation'):
    if key in res.observations:
        print(f"[{key}]\n{res.observations[key]}\n")
(OUT_DIR / 'observations.json').write_text(json.dumps(res.observations, indent=2))
print('saved observations.json')

# Extended causal analyses

The single-layer transplant above is 1.0 *by construction* — it pins the patched
layer to the donor and says nothing about depth. The three analyses below are the
non-tautological causal measurements: how far the effect **propagates**, whether
it changes **continuously** with the query, and how ablation sensitivity varies
with **processing stage**.

We reuse the same hook infrastructure; only what we capture and sweep changes. To
keep runtime reasonable these default to a spread of layers and one representative
within-group pair — widen the knobs for a fuller sweep.

## D · Propagation ("ripple") analysis

Patch the recipient's **full** query at one layer, then measure **every** layer.
Earlier layers are untouched (transfer ≈ 0), the patched layer is the identity
(transfer = 1), and later layers reveal whether the "recipient became donor" effect
**survives downstream computation**. This is the real test of the query pathway's
causal reach.

Outputs: a transfer **heatmap** (patch layer × measurement layer), **decay curves**
vs distance downstream, and an **effective propagation depth** per patch layer.

In [ ]:
# full baseline across ALL layers (needed to score every measurement layer)
Ln = model.config.num_hidden_layers
baseline_all = P3.pool_levels(
    P3.run_baseline(model, prompts, layers=list(range(Ln)),
                    min_query_pos=CONFIG['MIN_QUERY_POS']))['L3_per_head']

# patch layers spread across depth, for one representative within-group pair
ripple_pair = P3.within_group_pairs(model)[0]
RIPPLE_STRIDE = max(1, Ln // 8)
ripple_sites = [(L, ripple_pair[0], ripple_pair[1]) for L in range(0, Ln, RIPPLE_STRIDE)]
print(f"ripple: patching {len(ripple_sites)} layers (pair {ripple_pair}), measuring all {Ln}")

ripple_df = P3.experiment_ripple(model, prompts, ripple_sites, directions='one',
                                 min_query_pos=CONFIG['MIN_QUERY_POS'])
ripple_l3 = P3.pool_levels(ripple_df)['L3_per_head']
ripple_tf = P3.ripple_transfer(baseline_all, ripple_l3, metric='bos_prob')
ripple_tf.to_csv(OUT_DIR / 'tables' / 'ripple_transfer.csv', index=False)

prop = P3.effective_propagation_depth(ripple_tf, threshold=0.5)
print('\neffective propagation depth (layers with transfer >= 0.5 downstream):')
display(prop)

In [ ]:
P3.plot_ripple_heatmap(ripple_tf, OUT_DIR / 'figures' / 'D_ripple_heatmap.png',
                       value='transfer', show=True)
P3.plot_ripple_decay(ripple_tf, OUT_DIR / 'figures' / 'D_ripple_decay.png',
                     value='transfer', show=True)

A steep decay means the query's influence is quickly overwritten downstream
(sink specialization is re-derived layer by layer); a shallow decay means a single
layer's query commits the head's sink behaviour for many layers. The heatmap's
lower triangle is ~0 by construction (earlier layers are untouched), the diagonal
is 1 (identity), and the upper triangle is the propagation signal.

## E · Partial query transplant

Interpolate the recipient's query toward the donor's,
$Q' = (1-\alpha)\,Q_{\text{recipient}} + \alpha\,Q_{\text{donor}}$, for
$\alpha \in \{0, 0.25, 0.5, 0.75, 1\}$, measuring at the patched layer. This tests
whether sink specialization changes **continuously** with the query representation
(a smooth 0→1 transfer) or only flips at the endpoints.

Outputs: transfer fraction, BOS logit, and sink score as functions of $\alpha$.

In [ ]:
partial_pairs = P3.within_group_pairs(model)[:4]        # a few pairs
partial_layer = Ln // 2
partial_sites = [(partial_layer, a, b) for (a, b) in partial_pairs]

partial_df = P3.experiment_partial(model, prompts, partial_sites,
                                   alphas=(0.0, 0.25, 0.5, 0.75, 1.0),
                                   directions='one', min_query_pos=CONFIG['MIN_QUERY_POS'])
partial_c = P3.partial_curves(baseline_all, partial_df)
partial_c.to_csv(OUT_DIR / 'tables' / 'partial_transplant.csv', index=False)

display(partial_c.groupby('alpha')[['transfer', 'sink_score', 'bos_logit']]
        .agg(['mean', 'std']).round(4))
P3.plot_partial(partial_c, OUT_DIR / 'figures' / 'E_partial_transplant.png', show=True)

A transfer curve tracking the dashed $y=\alpha$ line means the mapping from
query to sink behaviour is close to **linear** in the query; concavity or convexity
shows where the QK-norm and softmax bend the relationship.

## F · Layer-wise ablation profiles

Rather than one averaged bar chart, run each ablation (zero / scale / normalize /
noise) at **every depth** and plot its effect against layer. This reveals whether
sensitivity to query perturbations depends on processing stage — e.g. whether early
layers are robust and late layers fragile, or vice versa.

Recall (from the notebook's opening) that pre-norm **scaling is inert** — QK-norm is
scale-invariant — so those curves should sit near 1.0 at all depths, a built-in
control. Zero, normalize, and noise are the informative interventions.

In [ ]:
ABLATION_STRIDE = max(1, Ln // 14)
profile_head = P3.within_group_pairs(model)[0][0]
profile_targets = [(L, profile_head) for L in range(0, Ln, ABLATION_STRIDE)]
print(f"ablation profiles: head {profile_head} at {len(profile_targets)} layers "
      f"x 5 interventions")

profile_ab = P3.experiment_ablation(model, prompts, profile_targets,
                                    min_query_pos=CONFIG['MIN_QUERY_POS'])
profile_ab_l3 = P3.pool_levels(profile_ab)['L3_per_head']
profiles = P3.ablation_profiles(baseline_all, profile_ab_l3, relative=True)
profiles.to_csv(OUT_DIR / 'tables' / 'ablation_profiles.csv', index=False)

P3.plot_ablation_profiles(profiles, OUT_DIR / 'figures' / 'F_ablation_profiles.png',
                          value='relative', show=True)
display(profiles.groupby('condition')['relative'].agg(['mean', 'min', 'max']).round(3))

Curves far from 1.0 mark depths where the sink is fragile to that
perturbation; curves near 1.0 mark robustness. The contrast between direction-
destroying ablations (zero / noise) and direction-preserving ones (scale /
normalize) localizes *where* in the network the query's **direction** carries the
sink signal.

## Part N · Download

In [ ]:
import shutil, tempfile
zip_base = Path(tempfile.mkdtemp()) / 'phase3_3_causal'
shutil.make_archive(str(zip_base), 'zip', OUT_DIR)
zip_path = str(zip_base) + '.zip'
print('zipped ->', zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
else:
    print('not in Colab; artifacts are under', OUT_DIR)